In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from scipy.spatial import cKDTree
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA


def reorient_patch_to_flat(patch_data):
    """
    rotate patch so local surface is horizontal (normal points up).
    uses pca to find the best-fit plane and rotates to align it with the xy plane.
    """
    xyz = patch_data[['X', 'Y', 'Z']].values
    pca = PCA(n_components=3)
    pca.fit(xyz)
    normal = pca.components_[2]
    z_axis = np.array([0, 0, 1])
    rotation_axis = np.cross(normal, z_axis)
    rotation_axis_norm = np.linalg.norm(rotation_axis)

    if rotation_axis_norm < 1e-6:
        rotation_matrix = np.eye(3)
    else:
        rotation_axis = rotation_axis / rotation_axis_norm
        cos_angle = np.dot(normal, z_axis)
        angle = np.arccos(np.clip(cos_angle, -1, 1))
        K = np.array([
            [0, -rotation_axis[2], rotation_axis[1]],
            [rotation_axis[2], 0, -rotation_axis[0]],
            [-rotation_axis[1], rotation_axis[0], 0]
        ])
        rotation_matrix = (np.eye(3) +
                           np.sin(angle) * K +
                           (1 - np.cos(angle)) * K @ K)

    xyz_rotated = xyz @ rotation_matrix.T
    xyz_centered = xyz_rotated - xyz_rotated.mean(axis=0)
    return xyz_centered[:, 0], xyz_centered[:, 1]


# paths
root_dir = Path("PATH")

anvil_dirs = [d for d in root_dir.iterdir() if d.is_dir()]
print(f"Found {len(anvil_dirs)} anvil(s)")

# colormap and patch count
cmap = 'viridis'
n_patches = 10

for anvil_dir in anvil_dirs:
    anvil_id = anvil_dir.name
    print(f"\nProcessing anvil: {anvil_id}")

    # load files
    spatial_join_files = list(anvil_dir.glob("*_spatial_join.txt"))
    output_files       = list(anvil_dir.glob("*_output.txt"))

    if not spatial_join_files:
        print("no spatial_join file found – skipping")
        continue
    if not output_files:
        print("no output file found – skipping")
        continue

    spatial_df = pd.read_csv(spatial_join_files[0], sep=';', skiprows=1,
                             names=['X', 'Y', 'Z', 'R', 'G', 'B',
                                    'roughness_temp', 'curvature_temp',
                                    'Nx', 'Ny', 'Nz'])

    output_df = pd.read_csv(output_files[0], sep=';', skiprows=1,
                            names=['X', 'Y', 'Z', 'roughness', 'curvature', 'condition'])
    output_df['condition'] = output_df['condition'].str.strip().str.lower()

    print(f"  spatial join: {len(spatial_df)} pts  |  output: {len(output_df)} pts")

    # merge rgb into output points
    damaged_output   = output_df[output_df['condition'] == 'damaged'].copy()
    undamaged_output = output_df[output_df['condition'] == 'undamaged'].copy()

    undamaged_merged = undamaged_output.merge(
        spatial_df[['X', 'Y', 'Z', 'R', 'G', 'B']],
        on=['X', 'Y', 'Z'], how='inner')

    spatial_xyz = spatial_df[['X', 'Y', 'Z']].values
    damaged_xyz = damaged_output[['X', 'Y', 'Z']].values
    tree        = cKDTree(spatial_xyz)
    distances, indices = tree.query(damaged_xyz, k=1)
    damaged_output['R'] = spatial_df.iloc[indices]['R'].values
    damaged_output['G'] = spatial_df.iloc[indices]['G'].values
    damaged_output['B'] = spatial_df.iloc[indices]['B'].values

    merged_df    = pd.concat([damaged_output, undamaged_merged], ignore_index=True)
    damaged_df   = merged_df[merged_df['condition'] == 'damaged']
    undamaged_df = merged_df[merged_df['condition'] == 'undamaged'].copy()

    # cluster undamaged points into patches
    kmeans = KMeans(n_clusters=n_patches, random_state=42)
    undamaged_df['patch_id'] = kmeans.fit_predict(undamaged_df[['X', 'Y', 'Z']])
    patch_centers = kmeans.cluster_centers_

    # downsample background for faster plotting
    sample_idx = np.random.choice(len(spatial_df),
                                  size=int(len(spatial_df) * 0.3),
                                  replace=False)
    spatial_sample = spatial_df.iloc[sample_idx]
    rgb_colors = np.column_stack([
        spatial_sample['R'].values / 255.0,
        spatial_sample['G'].values / 255.0,
        spatial_sample['B'].values / 255.0,
    ])

    # percentile colour limits
    rough_vmin = np.percentile(merged_df['roughness'], 5)
    rough_vmax = np.percentile(merged_df['roughness'], 95)
    curv_vmin  = np.percentile(merged_df['curvature'], 5)
    curv_vmax  = np.percentile(merged_df['curvature'], 95)

    # single figure, 1 column x 2 rows (roughness top, curvature bottom)
    fig = plt.figure(figsize=(18, 22))
    outer_gs = GridSpec(2, 1, figure=fig, hspace=0.10)

    metrics = [
        ('roughness', rough_vmin, rough_vmax, 'Roughness',      'Roughness [mm]'),
        ('curvature', curv_vmin,  curv_vmax,  'Mean curvature',  'Mean curvature'),
    ]

    for row_idx, (metric, vmin, vmax, title_sfx, cbar_lbl) in enumerate(metrics):

        # inner grid: 5 rows x 5 cols
        #   cols 0-1: main anvil (rowspan 5)
        #   cols 2-3: 10 patches (5 x 2)
        #   col  4:   colorbar (rowspan 5)
        inner_gs = GridSpecFromSubplotSpec(
            5, 5,
            subplot_spec=outer_gs[row_idx],
            wspace=0.02,
            hspace=0.30,
            width_ratios=[1, 1, 0.45, 0.45, 0.06],
        )

        # main anvil
        ax_main = fig.add_subplot(inner_gs[:, :2])
        ax_main.set_title(f"{anvil_id} – {title_sfx}", fontsize=14)

        ax_main.scatter(spatial_sample['X'], spatial_sample['Y'],
                        c=rgb_colors, s=0.2, alpha=1.0)

        if len(damaged_df) > 0:
            ax_main.scatter(damaged_df['X'], damaged_df['Y'],
                            c=damaged_df[metric], cmap=cmap,
                            s=1, alpha=0.7, edgecolors='none',
                            vmin=vmin, vmax=vmax)

        # 3x3 grid: 2 vertical + 2 horizontal evenly-spaced lines
        for x in np.linspace(spatial_sample['X'].min(), spatial_sample['X'].max(), 4)[1:-1]:
            ax_main.axvline(x, color='gold', linewidth=1.5, linestyle='-', alpha=0.8)
        for y in np.linspace(spatial_sample['Y'].min(), spatial_sample['Y'].max(), 4)[1:-1]:
            ax_main.axhline(y, color='gold', linewidth=1.5, linestyle='-', alpha=0.8)

        # undamaged patch numbers on main anvil
        for i, center in enumerate(patch_centers):
            ax_main.text(center[0], center[1], str(i + 1),
                         fontsize=9, ha='center', va='center',
                         color='gold', fontweight='bold')

        ax_main.set_xlabel('X (mm)')
        ax_main.set_ylabel('Y (mm)')
        ax_main.set_aspect('equal')
        ax_main.set_xlim(spatial_sample['X'].min() - 10,
                         spatial_sample['X'].max() + 10)
        ax_main.set_ylim(spatial_sample['Y'].min() - 10,
                         spatial_sample['Y'].max() + 10)

        # undamaged patch mini-panels
        for patch_idx in range(n_patches):
            patch_data = undamaged_df[undamaged_df['patch_id'] == patch_idx].copy()
            r = patch_idx // 2      # row 0-4
            c = 2 + (patch_idx % 2) # col 2 or 3

            ax_p = fig.add_subplot(inner_gs[r, c])

            if len(patch_data) > 0:
                x_flat, y_flat = reorient_patch_to_flat(patch_data)
                ax_p.scatter(x_flat, y_flat,
                             c=patch_data[metric].values, cmap=cmap,
                             s=5, edgecolors='none',
                             vmin=vmin, vmax=vmax)
                ax_p.set_aspect('equal')

            # label sits above the axes frame via set_title, never overlaps data
            ax_p.margins(y=0.15)
            ax_p.set_title(str(patch_idx + 1), fontsize=11, pad=4, color='black')
            ax_p.axis('off')

        # colorbar spanning full height of patch panel
        cax = fig.add_subplot(inner_gs[:, 4])
        sm  = plt.cm.ScalarMappable(
            cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
        sm.set_array([])
        cb = fig.colorbar(sm, cax=cax)
        cb.set_label(cbar_lbl, rotation=270, labelpad=16)

    # save
    output_path = anvil_dir / f"{anvil_id}_panel_layout.png"
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Saved: {output_path.name}")
    plt.close()

print("\nprocessing complete!")